In [ ]:
import numpy as np
import pandas as pd
import h5py
import warnings
warnings.filterwarnings("ignore")
from ReportPretraining import ReportClassification
from src.layers.transformer.lightcurve import LightCurveTransformer
from src.utils.data.AlerceDictionaries import ELASTICC_TAXONOMY, ZTF_TAXONOMY
import seaborn as sns
dataset  ='validation'
DEVICE = 'cuda:0'
import matplotlib.pyplot as plt
test_1 = ReportClassification(
    path_to_training_dir='/home/mdelafuente/pipeline/pipeline/training/lc_classifier_ztf/ATAT_ALeRCE/results/200/LC/class_alerce_baseline_0_sampler/',
    path_to_dataset= 'data/datasets/ZTF_ff/final/LC_MD_FEAT_240627_windows_200_12/dataset.h5',
    #path_to_dataset= '/home/mdelafuente/ZTF_SSL_Dataset/data/H5_FILES/ztf_ff_raw/h5_dataset/ztf_ff_raw.h5',
    #path_to_dataset= '/home/mdelafuente/ORIGINAL/elasticc_dataset_update.h5',
    model_class = LightCurveTransformer,
    custom_parse_key_str= '',
    model_type = 'lc',
    taxonomy = ZTF_TAXONOMY, #ELASTICC_TAXONOMY,
    device = DEVICE,
    seed = 0,
    figsize = (10,10),
    batch_size=16, 
    umap_args={"n_neighbors": 15, "min_dist": 0.05, "metric": "euclidean"},
    marker_size = 6)

In [ ]:
import torch
permutation = torch.random.permutations(iterable, r=None)(torch.arange(10), dims = (-1,))
print(permutation)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [ ]:
import matplotlib.pyplot as plt
test_1 = ReportClassification(
    path_to_training_dir='/home/mdelafuente/pipeline/pipeline/training/lc_classifier_ztf/ATAT_ALeRCE/results/200/LC/class_alerce_baseline_0_sampler/',
    path_to_dataset= 'data/datasets/ZTF_ff/final/LC_MD_FEAT_240627_windows_200_12/dataset.h5',
    #path_to_dataset= '/home/mdelafuente/ZTF_SSL_Dataset/data/H5_FILES/ztf_ff_raw/h5_dataset/ztf_ff_raw.h5',
    #path_to_dataset= '/home/mdelafuente/ORIGINAL/elasticc_dataset_update.h5',
    model_class = LightCurveTransformer,
    custom_parse_key_str= '',
    model_type = 'lc',
    taxonomy = ZTF_TAXONOMY, #ELASTICC_TAXONOMY,
    device = DEVICE,
    seed = 0,
    figsize = (10,10),
    batch_size=16, 
    umap_args={"n_neighbors": 15, "min_dist": 0.05, "metric": "euclidean"},
    marker_size = 6)


['/home/mdelafuente/pipeline/pipeline/training/lc_classifier_ztf/ATAT_ALeRCE/results/AUGS/LC/class_baseline/classifier_ckpt_step=100.ckpt']
using checkpoint 100.ckpt


RuntimeError: Error(s) in loading state_dict for ClassifierBaseModel:
	Unexpected key(s) in state_dict: "classifier.token_lc.norm.weight", "classifier.token_lc.norm.bias", "classifier.token_lc.output_layer.0.weight", "classifier.token_lc.output_layer.0.bias". 

: 

In [ ]:
dict_ = {
    "AGN": 0,
    "QSO": 1,
    "EA": 2,
    "YSO": 3,
    "SNIa": 4,
    "CV/Nova": 5,
    "RRLc": 6,
    "RSCVn": 7,
    "Blazar": 8,
    "SNII": 9,
    "EB/EW": 10,
    "LPV": 11,
    "CEP": 12,
    "RRLab": 13,
    "Periodic-Other": 14,
    "DSCT": 15,
    "SNIbc": 16,
    "SLSN": 17,
    "TDE": 18,
    "SNIIb": 19,
    "SNIIn": 20,
    "Microlensing": 21
}

dict_  = {value:key for key, value in dict_.items()}


In [ ]:
probs, target = test_1.predict_probs(dataset, modality =  "LC")

In [ ]:
binary = abs(np.argmax(probs, axis = -1) - target) > 0 # if zero, entonces le achunto
df = pd.DataFrame({'binary':binary, 'target':target.astype(int), 'predicted':np.argmax(probs, axis = -1)})
df['target_class_name'] = df['target'].map(dict_)
df['predicted_class_name'] = df['predicted'].map(dict_)

In [ ]:
with h5py.File('{}'.format('/home/mdelafuente/pipeline/pipeline/training/lc_classifier_ztf/ATAT_ALeRCE/data/datasets/ZTF_ff/final/LC_MD_FEAT_240627_windows_200_12/dataset.h5'), 'r') as f:
    these_idx = f.get('validation_0')
    print(f.get('flux'))
    count = np.count_nonzero(f.get('flux')[these_idx],axis = (1,2))
    print(count.shape)

In [ ]:
df['obs_count'] = count
to_be_removed = df.sort_values('obs_count').query('obs_count < 6').index.tolist()

In [ ]:
len(to_be_removed)

In [ ]:
from dataclasses import dataclass
import requests

import sqlalchemy as sa


@dataclass
class GLOBALDATA:
    OLD_DATASET_PATH = '/home/mdelafuente/pipeline/pipeline/training/lc_classifier_ztf/ATAT_ALeRCE/data/datasets/ZTF_ff/final/LC_MD_FEAT_240627_windows_200_12/dataset.h5'
    NEW_DATSET_PATH = '/home/mdelafuente/pipeline/pipeline/training/lc_classifier_ztf/ATAT_ALeRCE/TEST_NEW_DATASET.h5'


    TRAIN_KEY_NAME = 'training' 
    VAL_KEY_NAME = 'validation'
    TEST_KEY_NAME = 'test'

    OID_KEY_NAME = 'oid'

    FLUX_KEY_NAME = 'flux'
    FLUX_ERR_KEY_NAME = 'flux_err'

    TIME_KEY_NAME = 'time'
    TIME_DETECTION_KEY_NAME = 'time_detection'
    TIME_PHOTOMETRY_KEY_NAME = 'time_photometry'

    MASK_KEY_NAME = 'mask'
    MASK_DETECTION_KEY_NAME = 'mask_detection'
    MASK_PHOTOMETRY_KEY_NAME = 'mask_photometry'

    METADATA_KEY_NAME = 'metadata_feat'
    FEATURES_KEY_NAME = 'extracted_features'

    LABELS_KEY = 'labels'
    START_MJD = 'start_mjd'
    END_MJD = 'end_mjd'
    
class ModifyZTFDataset(GLOBALDATA):
    def __init__(self):
        super().__init__()

    def find_full_photometry_mask(self, dataset_key):
        full_photometry_idx = []

        with h5py.File('{}'.format(self.OLD_DATASET_PATH), 'r') as f:
            for i in range(f.get(f'{dataset_key}').shape[0]):
                if np.sum(f.get(f'{dataset_key}')[i]) == 200:
                    if f.get('labels')[i] in [4,9,16,17,19,20]:
                        full_photometry_idx.append(i)
        return full_photometry_idx
    
    def find_low_dispersion_windows(self, dataset_keys):

            if isinstance(dataset_keys, list):
                for key in dataset_keys:
                    print("Exploring dataset '{}'".format(key))
                    with h5py.File('{}'.format(self.OLD_DATASET_PATH), 'r') as f:
                        #print(f.keys())
                        these_idx = f.get('{}'.format(key))
                        #print(these_idx)
                        data = f.get(self.FLUX_KEY_NAME)[these_idx]
                        std_1 = np.std(data[:,0],axis = (1))
                        std_2 = np.std(data[:,1],axis = (1))
                        labels = f.get('labels')[these_idx]
                        df = pd.DataFrame({'idx_in_dataset':these_idx,
                                           'std_1': std_1, 
                                           'std_2':std_2, 
                                           'max_val_1':np.max(data[:,0], axis = 1),
                                           'max_val_2':np.max(data[:,1], axis = 1),
                                           'min_val_1':np.min(data[:,0], axis = 1),
                                           'min_val_2':np.min(data[:,1], axis = 1),
                                           'range' :abs(np.max(data[:,1], axis = 1) - np.min(data[:,1], axis = 1)),
                                           'labels': labels}).sort_values(['range'])
                        df['classes'] = df['labels'].map(dict_)
                        return df
    def find_bad_windows(self, dataset_keys):

        if isinstance(dataset_keys, list):
            to_be_removed = []
            to_be_kept = []
            for key in dataset_keys:
                print("Exploring dataset '{}'".format(key))
                with h5py.File('{}'.format(self.OLD_DATASET_PATH), 'r') as f:
                    #print(f.keys())
                    these_idx = f.get('{}'.format(key))
                    #print(these_idx)
                    count = np.count_nonzero(f.get(self.FLUX_KEY_NAME)[these_idx],axis = (1,2))
                    #print(count.shape)
                    df = pd.DataFrame({'idx_in_dataset':these_idx,'obs_count': count})
                    to_be_removed += df.sort_values('obs_count').query('obs_count < 6')['idx_in_dataset'].tolist()
                   # display(df.sort_values('obs_count').query('obs_count < 6'))
                    
                    print(' -   Original dataset len {} --> New len of dataset is {}'.format(len(these_idx),len(these_idx)- len(to_be_removed)))
                    
                    to_be_kept += df.sort_values('obs_count').query('obs_count >= 6')['idx_in_dataset'].tolist()
                   # display(df.sort_values('obs_count').query('obs_count >= 6'))
            to_be_removed.sort()
            to_be_kept.sort()
            return to_be_removed, to_be_kept
            
    def initialize_dataset_shapes(self, new_dataset_len):
        arrays_to_initialize = []
        arrays_to_initialize+=[self.FLUX_KEY_NAME]
        arrays_to_initialize+=[self.FLUX_ERR_KEY_NAME]

        arrays_to_initialize+=[self.TIME_KEY_NAME]
        arrays_to_initialize+=[self.TIME_PHOTOMETRY_KEY_NAME]
        arrays_to_initialize+=[self.TIME_DETECTION_KEY_NAME]

        arrays_to_initialize+=[self.MASK_KEY_NAME]
        arrays_to_initialize+=[self.MASK_DETECTION_KEY_NAME]
        arrays_to_initialize+=[self.MASK_PHOTOMETRY_KEY_NAME]

        arrays_to_initialize+=[self.METADATA_KEY_NAME]
        arrays_to_initialize+=[self.FEATURES_KEY_NAME]

        arrays_to_initialize+=[self.LABELS_KEY]

        key_shape_dict = {}
        same_shape_keys = [self.FLUX_KEY_NAME,
                        self.FLUX_ERR_KEY_NAME,
                        self.TIME_KEY_NAME,
                        self.TIME_DETECTION_KEY_NAME,
                        self.TIME_PHOTOMETRY_KEY_NAME,
                        self.MASK_KEY_NAME,
                        self.MASK_DETECTION_KEY_NAME,
                        self.MASK_PHOTOMETRY_KEY_NAME]
        for key in arrays_to_initialize:
            if key in same_shape_keys:
                key_shape_dict.update({key:(new_dataset_len,200,2)})
            elif key == self.FEATURES_KEY_NAME:
                key_shape_dict.update({key:(new_dataset_len,181,1)})
            elif key == self.METADATA_KEY_NAME:
                key_shape_dict.update({key:(new_dataset_len,6,1)})
            if key == self.LABELS_KEY:
                key_shape_dict.update({key:(new_dataset_len,)})
        #print(arrays_to_initialize)
        print(key_shape_dict)
        return key_shape_dict
    
    def create_modded_dataset(self,idx_to_keep, dataset_key, seed = 0):
        with h5py.File('{}'.format(self.OLD_DATASET_PATH), 'r+') as f:
            new_name = f'modded_{dataset_key}{'_'+str(seed) if seed != "" else ""}'
            if new_name in f.keys():
                del f[new_name]
            f.create_dataset(new_name, data = idx_to_keep)


In [ ]:
modifier_tool = ModifyZTFDataset()
#df = modifier_tool.find_low_dispersion_windows(['modded_validation_0'])

In [ ]:
len_full_phot = modifier_tool.find_full_photometry_mask('mask_detection')
len_full_phot

In [ ]:
print(len(len_full_phot))

In [ ]:
with h5py.File('{}'.format(modifier_tool.OLD_DATASET_PATH), 'r') as f:
    sns.lineplot(f.get('flux')[210484])


In [ ]:
DATASET_BEING_MODDED = ['validation_0']

IDX_TO_REMOVE,IDX_TO_KEEP = modifier_tool.find_bad_windows(DATASET_BEING_MODDED)
assert set(IDX_TO_REMOVE).intersection(IDX_TO_KEEP) == set() #check no crosscontamination
#key_shape_pairs = modifier_tool.initialize_dataset_shapes(len(IDX_TO_KEEP))
print(len(IDX_TO_KEEP))

modifier_tool.create_modded_dataset(IDX_TO_KEEP,'validation', seed = 0)
 
DATASET_BEING_MODDED = ['training_0']


IDX_TO_REMOVE,IDX_TO_KEEP = modifier_tool.find_bad_windows(DATASET_BEING_MODDED)
assert set(IDX_TO_REMOVE).intersection(IDX_TO_KEEP) == set() #check no crosscontamination
#key_shape_pairs = modifier_tool.initialize_dataset_shapes(len(IDX_TO_KEEP))
print(len(IDX_TO_KEEP))

modifier_tool.create_modded_dataset(IDX_TO_KEEP,'training', seed = 0)

DATASET_BEING_MODDED = ['test']


IDX_TO_REMOVE,IDX_TO_KEEP = modifier_tool.find_bad_windows(DATASET_BEING_MODDED)
assert set(IDX_TO_REMOVE).intersection(IDX_TO_KEEP) == set() #check no crosscontamination
#key_shape_pairs = modifier_tool.initialize_dataset_shapes(len(IDX_TO_KEEP))
print(len(IDX_TO_KEEP))
modifier_tool.create_modded_dataset(IDX_TO_KEEP,'test', seed = '')

In [ ]:
with h5py.File('{}'.format(modifier_tool.OLD_DATASET_PATH), 'r+') as f:
    print(f.keys())

In [ ]:
import os
with h5py.File('{}'.format(modifier_tool.NEW_DATSET_PATH), 'w') as f:
    for key,shape in key_shape_pairs.items():
        
        assert isinstance(key,str), 'key {} is not string'.format(key)
        assert isinstance(shape,tuple), 'shape {} is not of type tuple'.format(shape)
        empty_dataset = np.empty(shape)

        
        f.create_dataset(key, data=empty_dataset)
        print(f"Added key '{key}' dataset shape {empty_dataset.shape} as a HDF5 root level key")
        assert key in list(f.keys())
    
filesize = os.path.getsize(modifier_tool.NEW_DATSET_PATH)
filesize = str(np.round(filesize/10**6, 2)) + ' MB' if np.round(filesize /
                                                                        10**6, 2) <= 1000 else str(np.round(filesize/10**9, 2)) + ' GB'
print(f'Estimated dataset size is {filesize}')

In [ ]:
for key in ['flux', 'flux_err', 'labels', 'mask', 'mask_detection', 'mask_photometry', 'metadata_feat', 'time', 'time_detection', 'time_photometry']:
    print("Filling key {}".format(key))
    with h5py.File('{}'.format(modifier_tool.OLD_DATASET_PATH), 'r') as f:
        for new_idx,old_idx in enumerate(IDX_TO_KEEP):
            #print(new_idx, old_idx)
            
            data = f.get(key)[old_idx]
            with h5py.File('{}'.format(modifier_tool.NEW_DATSET_PATH), 'r+') as ff:
                    ff.get('{}'.format(key))[new_idx, :, :] = data
            if new_idx  == len(IDX_TO_KEEP):
                break

In [ ]:
with h5py.File('{}'.format(modifier_tool.OLD_DATASET_PATH), 'r') as f:
    print( f.get('flux')[0])

In [ ]:
with h5py.File('{}'.format(modifier_tool.NEW_DATSET_PATH), 'r') as ff:
   print( ff.get('flux')[0] )

In [ ]:

def insert_modded_folds(dataset):
    h5_ = h5py.File("{}".format(NEW_DATSET_PATH))
    df2 = df.reset_index()
    new_idx = set(df2['index'].values) - set(IDX_TO_REMOVE)
    assert len(new_idx) == h5_.get('flux').shape[0]
    print(len(new_idx))
    with h5py.File('{}'.format(NEW_DATSET_PATH), 'w') as f:
        f.create_dataset(dataset, data=list(new_idx))
#insert_modded_folds('validation_0')

In [ ]:
#partamos por las supernovas
display(df.query('binary == True and target_class_name == "SNII"'))
idx_sn2 = df.query('binary == True and target_class_name == "SNII"').index.tolist()

In [ ]:
df.query('binary == True and target_class_name == "SNII"')['obs_count'].hist()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
objects_of_interest = h5_.get('flux')[idx_sn2]

In [ ]:
for i in range(len(objects_of_interest[:10])):
    sns.lineplot(objects_of_interest[i])
    plt.show()
